## Data cleaning and Preprocessing

In [ ]:
# adding necessary libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [ ]:
# opening the dataset
data_filepath = "train_data/group_01_train.csv"
data = pd.read_csv(data_filepath)
data

In [ ]:
data.info()

In [ ]:
# number of duplicated rows 
data.duplicated().sum()

In [ ]:
# delete duplicated rows
data = data.drop_duplicates()

In [ ]:
# here we look at what percentage of the columns are null
missing_values = data.isna().sum().sort_values(ascending=False)
missing_percentage = missing_values[missing_values!=0]/len(data)*100
missing_percentage

In [ ]:
# dropping columns with missing percentage of more than 20%
over_miss_cols = missing_percentage[missing_percentage > 20].index
data = data.drop(columns=over_miss_cols)
print("Dropped columns:")
print(list(over_miss_cols))

In [ ]:
pd.DataFrame({
    'Dtype': data.dtypes,
    'Missing Count': data.isna().sum()
}).query('`Missing Count` > 0').sort_values('Missing Count', ascending=False)

In [ ]:
# filling numeric columns with the mean of their columns
numeric_cols = data.select_dtypes(include='number').columns
data[numeric_cols] = data[numeric_cols].fillna(data[numeric_cols].mean())

In [ ]:
pd.DataFrame({
    'Dtype': data.dtypes,
    'Missing Count': data.isna().sum()
}).query('`Missing Count` > 0').sort_values('Missing Count', ascending=False)

In [ ]:
# checking how many rows will be removed if we drop rows with missing values 
data1 = data
before = len(data1)
data1 = data1.dropna()
after = len(data1)

print("Rows removed:", before - after)

In [ ]:
# only about 3% of the rows drop, its better to just drop them
data = data.dropna()

In [ ]:
data.info()

In [ ]:
data.nunique()

In [ ]:
# dropping f11 because the only value used in this column is is 'US'
data = data.drop(columns=['f11'])

In [ ]:
# dropping f7 because converting it to a numerical value would need LLMs and NLPs which are not discussed in this course
data = data.drop(columns=['f7'])

In [ ]:
# dropping f8 because of the same reason for the above cell
data = data.drop(columns=['f8'])

In [ ]:
data.nunique()

In [ ]:
# we draw boxplot of dutation betwen f2 and f13 (f2 - f13) and see if there is any
# meaningful relation between y and this feature.

data['f13'] = pd.to_datetime(data['f13'], format='mixed')
data['f2'] = pd.to_datetime(data['f2'], format='mixed')

data['duration_min'] = (data['f2'] - data['f13']).dt.total_seconds() / 60.0

df_clean = data[data['duration_min'] > 0].copy()

sns.boxplot(data=df_clean, x='y', y='duration_min', showfliers=False)
plt.title('Event Duration vs Target Label (y)')
plt.xlabel('Target Label (y)')
plt.ylabel('Duration (Minutes)')
plt.show()

In [ ]:
# we use f13 to extract hour, day of the week, and month of the record and save them in different columns.
# then we draw a histogram to see relationship between hour, and 4 columns which are Day/Night (f31, f32, f33, f34)

data['f13'] = pd.to_datetime(data['f13'], format='mixed')

data['hour'] = data['f13'].dt.hour
data['day_of_week'] = data['f13'].dt.dayofweek
data['month'] = data['f13'].dt.month

plt.figure(figsize=(10, 5))
sns.histplot(data=data, x='hour', hue='f31', multiple='dodge', bins=24, shrink=0.8)
plt.title('Relationship Between Extracted Hour and Column f31')
plt.xlabel('Hour of the Day (0-23)')
plt.ylabel('Accident Count')
plt.xticks(range(0, 24))
plt.show()

In [ ]:
# we drop these 4 columns becuase they show hight colinearity with `hour`, also dropping original f2, f13
data = data.drop(columns=['f31', 'f32', 'f33', 'f34'])

In [ ]:
# we draw boxplot of dutation betwen f2 and f13 (f2 - f13) and see if there is any
# meaningful relation between y and duration.

data['f13'] = pd.to_datetime(data['f13'], format='mixed')
data['f2'] = pd.to_datetime(data['f2'], format='mixed')

data['duration_min'] = (data['f2'] - data['f13']).dt.total_seconds() / 60.0

# we need to make sure we use this step later, if we decide to use this feature in the model, because of negative values.
df_clean = data[data['duration_min'] > 0].copy()

sns.boxplot(data=df_clean, x='y', y='duration_min', showfliers=False)
plt.title('Event Duration vs Target Label (y)')
plt.xlabel('Target Label (y)')
plt.ylabel('Duration (Minutes)')
plt.show()

In [ ]:
# we drop these f2 and f13 cause their information is extracted and they're not needed anymore.
data = data.drop(columns=['f2', 'f13'])
data.columns

In [ ]:
# frequency encoding for high cardinal columns
cols = ['f9', 'f10', 'f18', 'f21']

for col in cols:
    freq = data[col].value_counts()
    data[col] = data[col].map(freq)

In [ ]:
# ont-hot-oncoding for categorical dtypes
cols = ['f1', 'f12']

data = pd.get_dummies(
    data,
    columns=cols,
    dtype='int64'
)

In [ ]:
# columns standardization
num_cols = [
    col for col in data.select_dtypes(include='number').columns
    if col != 'y'
]

scaler = StandardScaler()

data[num_cols] = scaler.fit_transform(data[num_cols])
data[num_cols] = data[num_cols].round(2)

In [ ]:
# number of duplicate rows
data.duplicated().sum()

In [ ]:
# getting rid of duplicate rows
data = data.drop_duplicates()

In [ ]:
data.to_csv('train_data/preprocessed_train.csv', index=False)